# Peak based ELM metrics

This notebook demonstrates the algorithms for ELM metrics. The metrics are calculated based on the peak values of the ELM signal. The metrics are:


## Content
- [ ] import data
- [ ] plot ground truth data
- [ ] plot simple peak detection on windows of data
- [ ] 

In [ ]:
import torch
import logging
from src.data_loaders import FusionShotDataModule
from src.config import load_config_from_file
import pandas as pd

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

C = load_config_from_file('fm_toy', as_omega=True)

ds = FusionShotDataModule(**C.data)
ds.prepare_data()
ds.setup()

In [ ]:
C.data

In [ ]:
ds.test_shots


In [ ]:
import plotly.graph_objects as go
from plotly import colors as plt_colors
from plotly.subplots import make_subplots as plotly_make_subplots
from scipy.signal import butter, filtfilt

COLOR_SCALE = plt_colors.qualitative.Plotly

def plot_shot(df, shot_i = None, shot_num=None):
    if shot_i is not None:
        shotnums = df['ShotNum'].unique()
        shot_num = shotnums[shot_i]
    shot_df: pd.DataFrame = df[df['ShotNum'] == shot_num]
    fig = go.Figure()
    fig.update_layout(
        title=f"Plot {shot_num}",
        template='plotly_dark',
    )
    for i, col in enumerate(C.data.cols.x +C.data.cols.c):
        fig.add_trace(go.Scatter(x=shot_df.index, y=shot_df[col], mode='lines', name=col, line_color=COLOR_SCALE[i % 10]))

    # smooth versions
    fig.add_trace(go.Scatter(x=shot_df.index, y=shot_df["NBI"].rolling(100, min_periods=1, center=True).median(), mode='lines', name="NBI median", line_color=COLOR_SCALE[(i + 1) % 10]))
    fig.add_trace(go.Scatter(x=shot_df.index, y=shot_df["ECRH"].rolling(100, min_periods=1, center=True).median(), mode='lines', name="ECRH median", line_color=COLOR_SCALE[(i + 2) % 10]))
    fig.show()

plot_shot(ds.data, 201)



In [ ]:
for i, shot in enumerate(ds.test_shots):
    plot_shot(ds.data, shot_num=shot)
    if i > 10: break

In [ ]:
from scipy.image import butter, filtfilt
b, a = butter(2, 500, btype='low', fs=10000)
filtered = filtfilt(b, a, signal)

In [ ]:
def plot_shot_with_filters(df, shot_i = None, shot_num=None):
    if shot_i is not None:
        shotnums = df['ShotNum'].unique()
        shot_num = shotnums[shot_i]
    shot_df: pd.DataFrame = df[df['ShotNum'] == shot_num]
    fig = go.Figure()
    fig.update_layout(
        title=f"Plot {shot_num}",
        template='plotly_dark',
    )
    for i, col in enumerate(C.data.cols.x +C.data.cols.c):
        fig.add_trace(go.Scatter(x=shot_df.index, y=shot_df[col], mode='lines', name=col, line_color=COLOR_SCALE[i % 10]))

    # smooth versions
    b, a = butter(2, 50, btype='low', fs=10000)
    # filtered = filtfilt(b, a, shot_df["NBI"])
    fig.add_trace(go.Scatter(x=shot_df.index, y=filtfilt(b, a, shot_df["NBI"]), mode='lines', name="NBI filtered", line_color=COLOR_SCALE[(i + 3) % 10]))
    fig.add_trace(go.Scatter(x=shot_df.index, y=shot_df["NBI"].rolling(100, min_periods=1, center=True).median(), mode='lines', name="NBI median", line_color=COLOR_SCALE[(i + 1) % 10]))
    fig.add_trace(go.Scatter(x=shot_df.index, y=filtfilt(b, a, shot_df["ECRH"]), mode='lines', name="ECRH filtered", line_color=COLOR_SCALE[(i + 2) % 10]))
    fig.show()

plot_shot(ds.data, 201)